## 1. Configuración y carga de credenciales

Localizo el archivo `.env`, cargo el token de acceso a TMDB y defino
la cabecera de autorización y las carpetas del proyecto. Compruebo que
ambos se han encontrado sin mostrar la credencial. Los dos valores
`True` confirman que he encontrado el archivo y cargado un valor en
`token`. La validez del token se comprueba después, al consultar la API.

In [45]:
import calendar
import json
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv

#find_dotenv() Localiza el .env y devuelve su ruta
#usecwd usa la carpeta de trabajo actual como punto de partida
ruta_env = find_dotenv(usecwd=True)
if not ruta_env:
    raise FileNotFoundError("No se ha encontrado el archivo .env.")

# Cargamos el token del .env; si ya existía otra variable con ese nombre, gana la del .env
load_dotenv(ruta_env, override=True)
token = os.getenv("TMDB_TOKEN")
if not token or not token.strip():
    raise ValueError("Falta TMDB_TOKEN o está vacío en el archivo .env.")

# Cabecera de autorización y dirección base de la API

headers = {"Authorization": f"Bearer {token}"}
URL_BASE = "https://api.themoviedb.org/3"

# La raíz del proyecto es la carpeta que contiene el .env
raiz = Path(ruta_env).resolve().parent
carpeta_raw = raiz / "src" / "data" / "raw"
carpeta_proc = raiz / "src" / "data" / "processed"
carpeta_raw.mkdir(parents=True, exist_ok=True)
carpeta_proc.mkdir(parents=True, exist_ok=True)

# Mostramos True/False y nunca el token, para no dejarlo en la salida del notebook
print("Archivo .env encontrado:", bool(ruta_env))
print("Clave cargada:", bool(token))

Archivo .env encontrado: True
Clave cargada: True


## 2. Obtención del catálogo de géneros

Consulto los géneros cinematográficos de TMDB y guardo sus
identificadores y nombres en una tabla. Esta correspondencia
permitirá interpretar los géneros del listado de películas. También
guardo el catálogo en un archivo para reutilizarlo en el análisis.

In [46]:
# Solicitamos los nombres de los géneros en español
respuesta_generos = requests.get(
    f"{URL_BASE}/genre/movie/list",
    headers=headers,
    params={"language": "es-ES"},
    timeout=30,
)

# Detenemos la ejecución si recibimos un error HTTP
respuesta_generos.raise_for_status()

# Convertimos la lista de géneros en una tabla, la clave donde están las listas de géneros es genres
generos = pd.DataFrame(respuesta_generos.json()["genres"])

# Mostramos los identificadores y nombres de todos los géneros
display(generos)

# Contamos las filas: cada fila corresponde a un género
print("Número de géneros cinematográficos:", len(generos))

,id,name
0,28,Acción
1,12,Aventura
2,16,Animación
3,35,Comedia
4,80,Crimen
5,99,Documental
6,18,Drama
7,10751,Familia
8,14,Fantasía
9,36,Historia


Número de géneros cinematográficos: 19


El catálogo consultado devuelve 19 géneros cinematográficos. Cada
género tiene un identificador (`id`) y un nombre (`name`).

Una película puede pertenecer a varios géneros, por lo que estas
categorías no son excluyentes. Guardo el catálogo para reutilizarlo en
el análisis sin volver a escribirlo a mano.

In [47]:
generos.to_json(
    carpeta_proc / "generos.json", orient="records", force_ascii=False, indent=2
)
print("Catálogo guardado en:", (carpeta_proc / "generos.json").relative_to(raiz))

Catálogo guardado en: src\data\processed\generos.json


## 3. Prueba de consulta con Barbie

Utilizo Barbie (2023) como caso de prueba conocido porque está
dirigida por Greta Gerwig. Esto permitirá comprobar si el código
recupera correctamente sus datos y créditos de dirección.

Esta elección sirve para validar el procedimiento y no determina
qué películas se incluirán en el análisis completo.

### 3.1. Búsqueda del identificador

Busco la película por título y año de estreno. Reviso los resultados
para obtener el identificador de la película correcta en TMDB.

In [48]:
# Buscamos Barbie por título y año de estreno
respuesta_busqueda = requests.get(
    f"{URL_BASE}/search/movie",
    headers=headers,
    params={"query": "Barbie", "primary_release_year": 2023, "page": 1},
    timeout=30,
)

# Detenemos la ejecución si recibimos un error HTTP
respuesta_busqueda.raise_for_status()

# Creamos una tabla con los resultados y revisamos los datos de identificación
df_busqueda = pd.DataFrame(respuesta_busqueda.json()["results"])
display(df_busqueda[["id", "title", "original_title", "release_date"]])

,id,title,original_title,release_date
0,346698,Barbie,Barbie,2023-07-19
1,1081709,Black Barbie,Black Barbie,2023-03-11
2,1150753,Barbie: The Perfect Woman?,"Barbie, die perfekte Frau?",2023-07-21
3,1168783,Bad Barbie,Bad Barbie,2023-09-04
4,1546950,Barbie Hell,Barbie Hell,2023-12-09
5,1198262,The Barbie Story,The Barbie Story,2023-08-25
6,1165223,My First Barbie: Happy DreamDay,My First Barbie: Happy DreamDay,2023-02-01
7,1153241,Barbie and Me,Barbie and Me,2023-07-27
8,1093349,Barbie: Skipper and the Big Babysitting Adventure,Barbie: Skipper and the Big Babysitting Adventure,2023-03-03
9,1119092,Et Dieu créa Barbie,Et Dieu créa Barbie,2023-04-28


In [49]:
# Guardamos el ID de Barbie (2023), identificado en la tabla
pelicula_id = 346698

### 3.2. Consulta de los detalles y créditos

Una vez identificada Barbie (2023), utilizo su ID para consultar
sus detalles. Solicito los textos en español y añado los créditos
con `append_to_response="credits"`.

In [50]:
# Solicitamos los detalles de la película y sus créditos
respuesta_pelicula = requests.get(
    f"{URL_BASE}/movie/{pelicula_id}",
    headers=headers,
    params={"language": "es-ES", "append_to_response": "credits"},
    timeout=30,
)

# Comprobamos si la respuesta indica un error HTTP
respuesta_pelicula.raise_for_status()

# Interpretamos el contenido JSON como un diccionario de Python
pelicula = respuesta_pelicula.json()

# Confirmamos que hemos recuperado la película correcta
print("Título:", pelicula["title"])
print("Estreno:", pelicula["release_date"])

Título: Barbie
Estreno: 2023-07-19


### 3.3. Exploración de la respuesta

Reviso las claves disponibles en el diccionario de la película
y compruebo la estructura de sus créditos antes de extraer
la información de dirección.

In [51]:
# Mostramos los campos disponibles en los detalles de la película
print("Campos de la película:", list(pelicula.keys()))

# Mostramos los apartados incluidos en los créditos
print("Apartados de créditos:", list(pelicula["credits"].keys()))

Campos de la película: ['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'credits']
Apartados de créditos: ['cast', 'crew']


### 3.4. Exploración del equipo técnico

Convierto la lista `crew` en una tabla para revisar los nombres,
departamentos y puestos de las personas acreditadas.
Muestro únicamente las primeras cinco filas.

In [53]:
# Extraemos el equipo técnico y lo convertimos en una tabla
df_equipo = pd.DataFrame(pelicula["credits"]["crew"])

# Mostramos cinco filas y las columnas que queremos explorar
display(
    df_equipo.reindex(
        columns=["id", "name", "department", "job", "gender"]
    ).head(5)
)

,id,name,department,job,gender
0,36656,Sarah Greenwood,Art,Production Design,1
1,2399179,Michael Sharp,Production,Executive Producer,2
2,41080,Allison Jones,Production,Casting,1
3,23424,Lucy Bevan,Production,Casting,1
4,36658,Katie Spencer,Art,Set Decoration,1


### 3.5. Identificación de la dirección

Selecciono las filas del equipo técnico cuyo puesto (`job`) es
`Director`. Conservo todas las coincidencias porque una película
puede estar dirigida por varias personas.

In [54]:
# Seleccionamos los créditos cuyo puesto es Director
df_direccion = df_equipo.loc[
    df_equipo["job"] == "Director"
].copy()

# Mostramos los datos de las personas acreditadas en dirección
display(
    df_direccion.reindex(
        columns=["id", "name", "gender", "job"]
    )
)

,id,name,gender,job
179,45400,Greta Gerwig,1,Director


### 3.6. Interpretación del género de la persona

Relaciono los códigos de género con sus etiquetas. Conservo la columna
original `gender` y añado `genero_persona` para facilitar la lectura.
Este dato corresponde a la persona, no al género cinematográfico.

TMDB devuelve el género mediante códigos numéricos. Utilizo un
diccionario y el método `.map()` para crear una columna con etiquetas,
conservando el código original.

En esta prueba, Greta Gerwig aparece acreditada como directora y
registrada con el código 1 (mujer).

Los valores no especificados
se conservarán como información desconocida.

In [55]:
# Diccionario que relaciona cada código con su significado
etiquetas_genero = {
    0: "No especificado",
    1: "Mujer",
    2: "Hombre",
    3: "Persona no binaria"
}

# Creamos una columna nueva sin eliminar el código original
df_direccion["genero_persona"] = df_direccion["gender"].map(etiquetas_genero)

display(
    df_direccion[
        ["id", "name", "gender", "genero_persona", "job"]
    ]
)

,id,name,gender,genero_persona,job
179,45400,Greta Gerwig,1,Mujer,Director


### 3.7. Géneros cinematográficos de la película

Convierto la lista de géneros en una tabla para consultar sus
identificadores y nombres. Una película puede tener varios géneros.

In [56]:
# Extraemos los géneros de la película y creamos una tabla
df_generos_pelicula = pd.DataFrame(pelicula["genres"])

# Mostramos sus identificadores y nombres
display(df_generos_pelicula)

,id,name
0,35,Comedia
1,12,Aventura
2,14,Fantasía


### 3.8. Almacenamiento de la respuesta original

Guardo los detalles y créditos de Barbie en formato JSON dentro de
`src/data/raw`. Así puedo conservar los datos recibidos y volver
a consultarlos sin repetir la solicitud a la API.

In [57]:
# Construimos la ruta del archivo con el identificador de la película
ruta_archivo = carpeta_raw / f"pelicula_{pelicula_id}.json"

# Guardamos el diccionario completo en formato JSON
with open(ruta_archivo, "w", encoding="utf-8") as archivo:
    json.dump(pelicula, archivo, ensure_ascii=False, indent=4)

# Mostramos dónde se ha guardado
print("Archivo guardado en:", ruta_archivo.relative_to(raiz))

Archivo guardado en: src\data\raw\pelicula_346698.json


Ahora compruebo que puedo recuperar los datos desde el archivo, sin consultar la API.

In [58]:
# Abrimos el JSON en modo lectura
with open(ruta_archivo, "r", encoding="utf-8") as archivo:
    pelicula_guardada = json.load(archivo)

# Comprobamos qué película hemos recuperado
print("Identificador:", pelicula_guardada["id"])
print("Título:", pelicula_guardada["title"])

Identificador: 346698
Título: Barbie


## 4. Obtención del listado de películas

El objetivo es recuperar las películas disponibles en TMDB estrenadas
entre 2000 y 2025, excluyendo el contenido marcado como adulto
(`include_adult=false`). Este filtro no equivale a excluir las películas
clasificadas para mayores de 18 años.

No limitaré la selección a las primeras películas ni aplicaré filtros
previos por puntuación, popularidad, género cinematográfico o género
de la persona que dirige.

Primero consultaré una página para entender la estructura del listado
y su paginación. Después organizaré la descarga por intervalos de fechas.

### 4.1. Consulta de una página de prueba

Consulto las películas estrenadas en 2023 mediante `/discover/movie`.
Reviso el número de resultados y páginas antes de preparar la descarga.
Esta consulta inicial no constituye el conjunto completo del análisis.

Compruebo tres datos de la respuesta:

- **Cuántas películas coinciden** con las condiciones: `total_results`.
- **En cuántas páginas se reparten**: `total_pages`.
- **Qué películas vienen en la página solicitada**: `results`.

In [59]:
# Solicitamos la primera página de películas estrenadas en 2023
respuesta_listado = requests.get(
    f"{URL_BASE}/discover/movie",
    headers=headers,
    params={
        "primary_release_date.gte": "2023-01-01",
        "primary_release_date.lte": "2023-12-31",
        "include_adult": "false",
        "language": "es-ES",
        "page": 1,
    },
    timeout=30,
)

# Comprobamos si se ha producido un error HTTP
respuesta_listado.raise_for_status()

# Interpretamos el JSON como un diccionario de Python
datos_listado = respuesta_listado.json()

# Revisamos la información de paginación
print("Página recibida:", datos_listado["page"])
print("Total de películas:", datos_listado["total_results"])
print("Total de páginas:", datos_listado["total_pages"])
print("Películas en esta página:", len(datos_listado["results"]))

Página recibida: 1
Total de películas: 20001
Total de páginas: 1001
Películas en esta página: 20


### 4.1.1. Límite de paginación y estrategia de descarga

La consulta de 2023 indica 20.001 películas repartidas en 1.001 páginas.
Sin embargo, la documentación oficial de TMDB establece que solo se
pueden solicitar páginas de la 1 a la 500.

Fuente: [Documentación de errores de TMDB, código 22](https://developer.themoviedb.org/docs/errors).

Para evitar dejar resultados fuera por este límite, dividiré cada año
en meses y comprobaré el número de páginas de cada consulta. Si algún
mes supera las 500 páginas, lo dividiré en intervalos más pequeños.

### 4.2. Comprobación de un intervalo mensual

Consulto enero de 2023 y reviso cuántas páginas tiene. Esta comprobación
permite decidir si puedo descargar el mes completo o debo dividirlo
en intervalos más pequeños.

In [60]:
# Consultamos la primera página de enero de 2023
respuesta_mes = requests.get(
    f"{URL_BASE}/discover/movie",
    headers=headers,
    params={
        "primary_release_date.gte": "2023-01-01",
        "primary_release_date.lte": "2023-01-31",
        "include_adult": "false",
        "language": "es-ES",
        "page": 1,
    },
    timeout=30,
)

# Comprobamos la respuesta e interpretamos el JSON
respuesta_mes.raise_for_status()
datos_mes = respuesta_mes.json()

# Revisamos si el intervalo está dentro del límite
print("Películas en enero:", datos_mes["total_results"])
print("Páginas en enero:", datos_mes["total_pages"])
print("Dentro del límite:", datos_mes["total_pages"] <= 500)

Películas en enero: 4739
Páginas en enero: 237
Dentro del límite: True


La consulta de enero de 2023 indica 4.740 películas repartidas en
237 páginas. El intervalo está dentro del límite de 500 páginas,
por lo que no necesita subdividirse.

Hasta ahora solo he solicitado la primera página. Para recuperar
el listado completo del mes, tendré que recorrer las páginas 1 a 237.

### 4.3. Preparación de los intervalos mensuales

Creo una tabla con las fechas de inicio y fin de cada mes de 2023.
Estos intervalos permitirán consultar el año completo sin solapar fechas.

In [61]:
# Año que estamos utilizando para la prueba
anio = 2023

# Guardaremos un diccionario por cada mes
intervalos = []

# Recorremos los meses del 1 al 12
for mes in range(1, 13):

    # Obtenemos el número de días del mes
    ultimo_dia = calendar.monthrange(anio, mes)[1]

    # Guardamos las fechas de inicio y fin
    intervalos.append({
        "fecha_inicio": f"{anio}-{mes:02d}-01",
        "fecha_fin": f"{anio}-{mes:02d}-{ultimo_dia:02d}"
    })

# Convertimos los intervalos en una tabla
df_intervalos = pd.DataFrame(intervalos)

display(df_intervalos)

,fecha_inicio,fecha_fin
0,2023-01-01,2023-01-31
1,2023-02-01,2023-02-28
2,2023-03-01,2023-03-31
3,2023-04-01,2023-04-30
4,2023-05-01,2023-05-31
5,2023-06-01,2023-06-30
6,2023-07-01,2023-07-31
7,2023-08-01,2023-08-31
8,2023-09-01,2023-09-30
9,2023-10-01,2023-10-31


### 4.4. Comprobación de la paginación por mes

Consulto la primera página de cada intervalo mensual y registro el
total de películas y páginas. Identifico los meses que necesitan
subdividirse porque superan las 500 páginas.

In [62]:
# Guardaremos los resultados de la comprobación de cada mes
control_intervalos = []

# Recorremos los doce intervalos que hemos preparado
for intervalo in intervalos:
    respuesta_intervalo = requests.get(
        f"{URL_BASE}/discover/movie",
        headers=headers,
        params={
            "primary_release_date.gte": intervalo["fecha_inicio"],
            "primary_release_date.lte": intervalo["fecha_fin"],
            "include_adult": "false",
            "language": "es-ES",
            "page": 1,
        },
        timeout=30,
    )

    # Detenemos la ejecución si hay un error HTTP
    respuesta_intervalo.raise_for_status()
    datos_intervalo = respuesta_intervalo.json()

    # Registramos las fechas y los totales indicados por la API
    control_intervalos.append({
        "fecha_inicio": intervalo["fecha_inicio"],
        "fecha_fin": intervalo["fecha_fin"],
        "total_peliculas": datos_intervalo["total_results"],
        "total_paginas": datos_intervalo["total_pages"],
        "necesita_division": datos_intervalo["total_pages"] > 500,
    })

# Mostramos la tabla de control
df_control_intervalos = pd.DataFrame(control_intervalos)
display(df_control_intervalos)

,fecha_inicio,fecha_fin,total_peliculas,total_paginas,necesita_division
0,2023-01-01,2023-01-31,4739,237,False
1,2023-02-01,2023-02-28,2650,133,False
2,2023-03-01,2023-03-31,3926,197,False
3,2023-04-01,2023-04-30,3819,191,False
4,2023-05-01,2023-05-31,3904,196,False
5,2023-06-01,2023-06-30,4464,224,False
6,2023-07-01,2023-07-31,3261,164,False
7,2023-08-01,2023-08-31,3517,176,False
8,2023-09-01,2023-09-30,4696,235,False
9,2023-10-01,2023-10-31,5529,277,False


In [63]:
# Sumamos los resultados indicados para los doce meses
total_mensual = df_control_intervalos["total_peliculas"].sum()

# Comparamos con el total indicado en la consulta anual
print("Total según consulta anual:", datos_listado["total_results"])
print("Suma de los meses:", total_mensual)

Total según consulta anual: 20001
Suma de los meses: 49878


### 4.4.1. Comparación de los recuentos anual y mensual

La consulta anual de 2023 devuelve `total_results = 20001`, mientras
que la suma de las doce consultas mensuales devuelve 49.869 resultados.

Los recuentos no coinciden, por lo que no utilizaré el total anual
como referencia exacta para validar la descarga.

Todos los intervalos mensuales están por debajo de 500 páginas.
Tras descargarlos, comprobaré los registros recibidos y los
identificadores únicos para detectar posibles duplicados.

### 4.5. Función para consultar una página

Defino una función que recibe las fechas de inicio y fin y el número
de página. Realiza la solicitud, comprueba los errores HTTP y devuelve
la respuesta interpretada como un diccionario de Python. Esta misma
función se reutiliza después en el notebook de obtención de datos.

In [64]:
def consultar_pagina(fecha_inicio, fecha_fin, pagina=1):
    # Solicitamos una página del intervalo indicado
    respuesta = requests.get(
        "https://api.themoviedb.org/3/discover/movie",
        headers={"Authorization": f"Bearer {token}"},
        params={
            "primary_release_date.gte": fecha_inicio,
            "primary_release_date.lte": fecha_fin,
            "include_adult": "false",
            "language": "es-ES",
            "page": pagina
        },
        timeout=30
    )

    # Comprobamos los errores HTTP
    respuesta.raise_for_status()

    # Devolvemos los datos para utilizarlos fuera de la función
    return respuesta.json()

In [65]:
# Llamamos a la función para consultar la primera página de enero
pagina_prueba = consultar_pagina(
    fecha_inicio="2023-01-01",
    fecha_fin="2023-01-31",
    pagina=1
)

# Comprobamos la página recibida y cuántas películas contiene
print("Página recibida:", pagina_prueba["page"])
print("Películas recibidas:", len(pagina_prueba["results"]))

# Mostramos algunos datos de las primeras cinco películas
df_pagina_prueba = pd.DataFrame(pagina_prueba["results"])

display(
    df_pagina_prueba.reindex(
        columns=["id", "title", "release_date"]
    ).head(5)
)

Página recibida: 1
Películas recibidas: 20


,id,title,release_date
0,842675,La Tierra errante II,2023-01-22
1,739405,Operación Fortune: El gran engaño,2023-01-04
2,877703,Teen Wolf,2023-01-18
3,816904,Momias,2023-01-05
4,646389,El piloto,2023-01-11


In [66]:
# Lista donde acumularemos las películas recibidas
peliculas_prueba = []

# Consultamos las páginas 1 y 2
for numero_pagina in range(1, 3):
    datos_pagina = consultar_pagina(
        fecha_inicio="2023-01-01",
        fecha_fin="2023-01-31",
        pagina=numero_pagina
    )

    # Añadimos cada película de esta página a la lista
    peliculas_prueba.extend(datos_pagina["results"])

    print("Página recibida:", datos_pagina["page"])

print("Total de registros recibidos:", len(peliculas_prueba))

Página recibida: 1
Página recibida: 2
Total de registros recibidos: 40


In [67]:
# Convertimos las películas de las dos páginas en una tabla
df_prueba = pd.DataFrame(peliculas_prueba)

# Contamos los registros y los identificadores únicos
print("Registros recibidos:", len(df_prueba))
print("Películas únicas:", df_prueba["id"].nunique())
print("Identificadores duplicados:", df_prueba["id"].duplicated().sum())

Registros recibidos: 40
Películas únicas: 40
Identificadores duplicados: 0


### Conclusión de las pruebas

Las pruebas con la API permiten comprobar que:

- El token se carga sin mostrarse y las consultas se autentican bien.
- El catálogo contiene 19 géneros, que se guardan en `generos.json`.
- Los créditos permiten identificar a las personas con puesto `Director`
  y su código de género, como se comprobó con Barbie.
- Una consulta anual de 2023 indica 1.001 páginas, pero TMDB solo
  permite solicitar hasta la 500, y su total anual no coincide con la
  suma de los meses.
- Los meses de 2023 están por debajo de las 500 páginas, y la función
  descarga páginas sin duplicar identificadores en la prueba (40
  registros únicos).

En el notebook `01_obtencion_datos` aplico este procedimiento a los 312
meses entre 2000 y 2025.